# Knowledge Distillation (RKD-only): ConvNeXt V2 → MobileNetV3 (Colab)

**Experiment 27 — Bỏ L_KD_cosine, chỉ dùng RKD-D + RKD-A**

**Lý do thay đổi so với Exp26 (task=2 kd=1 rkd_d=1 rkd_a=2):**
- `loss_kd` epoch40 = 0.9837 → cosine similarity student↔teacher ≈ 0.016 (gần vuông góc)
- Nguyên nhân: L_MagFace (weight=2) và L_KD_cosine (weight=1) xung đột gradient
  vì student MagLinear.weight hội tụ về hướng KHÁC với teacher (random init ≠ teacher W)
- Rank-1 = 78% ≈ teacher 80% nhờ MagFace, không nhờ KD cosine
- RKD-D + RKD-A weight=1,2 → đóng góp 0.25% total loss → không học được gì

**Thay đổi:**
- `kd_weight = 0` → loại xung đột gradient
- `rkd_d_weight = 80` → ~8.5% task loss (meaningful regularization)
- `rkd_a_weight = 160` → ~12% task loss  (paper: rkd_a = 2×rkd_d)

| | Teacher | Student |
|---|---|---|
| Model | `MTLFaceRecognition` (ConvNeXt V2) | `FaceRecognitionMobileNetV3` |
| Params | ~28M | ~5M |
| Embedding | `x_id` (512-D) | 512-D |
| Mode | **Frozen** | **Trainable** |

**Loss:**
```
L_total = α·L_MagFace  +  γ·L_RKD_D  +  δ·L_RKD_A
        = 2·L_MagFace  +  80·L_RKD_D  +  160·L_RKD_A
```

## 1. Mount Drive & Setup môi trường

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

REPO_URL    = 'https://github.com/NguyenXuanBinh22/DATN.git'
REPO_BRANCH = 'convnext-v2-dev'
REPO_DIR    = '/content/FR_Photometric_Stereo'

if not os.path.exists(REPO_DIR):
    os.system(f'git clone -b {REPO_BRANCH} {REPO_URL} {REPO_DIR}')
else:
    os.system(f'git -C {REPO_DIR} pull origin {REPO_BRANCH}')
    print('Repo đã tồn tại, đã pull latest.')

%cd {REPO_DIR}
print(f'Working dir: {os.getcwd()}')

os.system('pip install -q albumentations==1.3.1 timm tabulate termcolor')

## 2. Imports & Cấu hình

In [ ]:
%cd /content/FR_Photometric_Stereo
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import albumentations as A
from torch.optim import Adam
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts
from torch.utils.tensorboard import SummaryWriter
from tabulate import tabulate

from going_modular.dataloader.multitask import create_multitask_datafetcher, create_eval_loaders
from going_modular.model.MTLFaceRecognition import MTLFaceRecognition
from going_modular.model.FaceRecognitionMobileNetV3 import FaceRecognitionMobileNetV3
from going_modular.loss.WeightClassMagLoss import WeightClassMagLoss
from going_modular.utils.transforms import RandomResizedCropRect, GaussianNoise
from going_modular.utils.roc_auc_id import (
    compute_id_auc, compute_rank1,
    compute_id_auc_gallery_probe, compute_rank1_gallery_probe,
)
from going_modular.utils.MultiMetricEarlyStopping import MultiMetricEarlyStopping
from going_modular.utils.ModelCheckPoint import ModelCheckpoint
from going_modular.utils.ExperimentManager import ExperimentManager

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(42)
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# ════════════════════════════════════════════════════════════
#  CẤU HÌNH — Experiment 27: RKD-only (bỏ L_KD_cosine)
# ════════════════════════════════════════════════════════════

DRIVE_DATASET_DIR = '/content/drive/MyDrive/Photometric_DB_Full/'

TEACHER_CKPT = '/content/drive/MyDrive/Photometric_DB_Full/experiments/Single_ALBEDO_PK_SAMPLER_ConvNextV2/checkpoints/best_model.pth'

EXPERIMENT_NAME = 'KD_RKD_ConvNextV2_to_MobileNetV3_Albedo_27_noKDcosine'

CONFIGURATION = {
    'note':        EXPERIMENT_NAME,
    'dataset_dir': DRIVE_DATASET_DIR,
    'output_dir':  '/content/drive/MyDrive/',

    'type':        'albedo',

    'teacher_backbone': 'convnextv2_tiny',
    'backbone':         'mobilenetv3_large_100',

    'use_sampler': True,
    'device':      device,
    'epochs':      40,
    'batch_size':  32,
    'image_size':  112,
    'base_lr':     1e-4,
    'num_classes': None,

    # ── Thay đổi chính so với Exp26 ──────────────────────────────────────
    # Exp26 (task=2 kd=1 rkd_d=1 rkd_a=2) → rank-1=78%, loss_kd không giảm
    # Vấn đề: L_MagFace vs L_KD_cosine xung đột gradient → kd_weight=0
    # RKD weights tăng để compensate (raw values epoch40: rkd_d≈0.007, rkd_a≈0.005)
    #
    # Ước tính đóng góp vào total loss khi converged:
    #   task  : 2.0 × 3.08 ≈ 6.16  (~79%)
    #   rkd_d : 80  × 0.007 ≈ 0.56  (~7%)
    #   rkd_a : 160 × 0.005 ≈ 0.80  (~10%)
    #   kd    : 0   × ...   = 0
    'task_weight':  2.0,
    'kd_weight':    0.0,    # ← BỎ L_KD_cosine (gây xung đột gradient)
    'rkd_d_weight': 80.0,   # ← tăng từ 1 → 80: ~7% task
    'rkd_a_weight': 160.0,  # ← tăng từ 2 → 160: ~10% task (paper: 2×rkd_d)
}

print(f"Dataset dir : {CONFIGURATION['dataset_dir']}")
print(f"Output dir  : {CONFIGURATION['output_dir']}")
print(f"Teacher ckpt: {TEACHER_CKPT}")
print(f"L_KD_cosine : {'DISABLED' if CONFIGURATION['kd_weight'] == 0 else f\"weight={CONFIGURATION['kd_weight']}\"}")
print(f"RKD weights : D={CONFIGURATION['rkd_d_weight']}  A={CONFIGURATION['rkd_a_weight']}")

# Ước tính đóng góp (extrapolate từ raw values Exp26)
tw = CONFIGURATION['task_weight'];   kw = CONFIGURATION['kd_weight']
dw = CONFIGURATION['rkd_d_weight'];  aw = CONFIGURATION['rkd_a_weight']
c_task = tw * 3.08;  c_kd = kw * 0.98;  c_d = dw * 0.007;  c_a = aw * 0.005
total  = c_task + c_kd + c_d + c_a
print(f"\nƯớc tính đóng góp khi converged (extrapolate Exp26):")
print(f"  task   : {c_task:.2f}  ({100*c_task/total:.1f}%)")
print(f"  kd     : {c_kd:.2f}  ({100*c_kd/total:.1f}%)")
print(f"  rkd_d  : {c_d:.3f}  ({100*c_d/total:.1f}%)")
print(f"  rkd_a  : {c_a:.3f}  ({100*c_a/total:.1f}%)")

## 3. Data Loading

In [ ]:
dataset_dir = CONFIGURATION['dataset_dir']

train_csv = os.path.join(dataset_dir, 'train_split.csv')
if not os.path.exists(train_csv):
    train_csv = os.path.join(dataset_dir, 'dataset', 'train_split.csv')
if not os.path.exists(train_csv):
    train_csv = os.path.join(dataset_dir, 'train_set.csv')
if not os.path.exists(train_csv):
    raise FileNotFoundError(
        f'Không tìm thấy CSV train tại {dataset_dir}.\n'
        f'Kiểm tra lại DRIVE_DATASET_DIR.'
    )
print(f'Train CSV: {train_csv}')

df_train = pd.read_csv(train_csv)
CONFIGURATION['num_classes'] = int(df_train['id'].nunique())
print(f'num_classes : {CONFIGURATION["num_classes"]}')
print(f'Số mẫu train: {len(df_train)}')

train_transform = A.Compose([
    RandomResizedCropRect(CONFIGURATION['image_size']),
    GaussianNoise(p=0.2),
    A.HorizontalFlip(p=0.5),
])
test_transform = A.Compose([
    A.Resize(CONFIGURATION['image_size'], CONFIGURATION['image_size']),
])

train_dl, test_dl, _ = create_multitask_datafetcher(
    CONFIGURATION, train_transform, test_transform, 'train_split.csv', 'probe_split.csv'
)
print(f'Train batches: {len(train_dl)} | Test batches (probe): {len(test_dl)}')

gallery_dl, probe_dl = create_eval_loaders(CONFIGURATION, test_transform)

## 4. Teacher Model (ConvNeXt V2 — Frozen)

In [ ]:
if not os.path.exists(TEACHER_CKPT):
    raise FileNotFoundError(
        f'Không tìm thấy teacher checkpoint: {TEACHER_CKPT}\n'
        f'Kiểm tra lại biến TEACHER_CKPT.'
    )

teacher = MTLFaceRecognition(
    backbone=CONFIGURATION['teacher_backbone'],
    num_classes=CONFIGURATION['num_classes'],
)

ckpt = torch.load(TEACHER_CKPT, map_location=device)
teacher.load_state_dict(ckpt)
print(f"Teacher loaded — epoch {ckpt.get('epoch', '?')}")

teacher.to(device)
teacher.eval()
for p in teacher.parameters():
    p.requires_grad = False

teacher_params = sum(p.numel() for p in teacher.parameters())
print(f'Teacher params: {teacher_params:,} (tất cả frozen)')

with torch.no_grad():
    _dummy = torch.randn(2, 3, 112, 112).to(device)
    _t_emb = teacher.get_embedding(_dummy)[-1]
    print(f'Teacher ID embedding shape: {_t_emb.shape}')

## 4.1. Đánh giá Teacher Model (baseline)

Đo AUC và Rank-1 của teacher trước distillation để so sánh với student sau này.

In [ ]:
class _TeacherEvalWrapper(torch.nn.Module):
    """MTLFaceRecognition.get_embedding() trả về tuple — wrapper giữ lại x_id (index -1)."""
    def __init__(self, teacher):
        super().__init__()
        self._teacher = teacher

    def get_embedding(self, x):
        return self._teacher.get_embedding(x)[-1]


teacher_wrapper = _TeacherEvalWrapper(teacher).to(device)
teacher_wrapper.eval()

teacher_gp_auc   = compute_id_auc_gallery_probe(gallery_dl, probe_dl, teacher_wrapper, device)
teacher_gp_rank1 = compute_rank1_gallery_probe(gallery_dl, probe_dl, teacher_wrapper, device)

rows = [
    ['Cosine AUC    (gallery→probe)', f"{teacher_gp_auc['id_cosine']:.4f}"],
    ['Euclidean AUC (gallery→probe)', f"{teacher_gp_auc['id_euclidean']:.4f}"],
    ['Rank-1 Acc    (gallery→probe)', f"{teacher_gp_rank1:.4f}"],
]
print(f"Teacher ({CONFIGURATION['teacher_backbone']}) — {TEACHER_CKPT}")
print(tabulate(rows, headers=['Metric', 'Value'], tablefmt='fancy_grid'))

## 5. Student Model (MobileNetV3 — Trainable)

In [ ]:
student = FaceRecognitionMobileNetV3(
    num_classes=CONFIGURATION['num_classes'],
    backbone=CONFIGURATION['backbone'],
)
student.to(device)

total_p     = sum(p.numel() for p in student.parameters())
trainable_p = sum(p.numel() for p in student.parameters() if p.requires_grad)
print(f'Student total params    : {total_p:,}')
print(f'Student trainable params: {trainable_p:,}')

with torch.no_grad():
    _s_emb = student.get_embedding(_dummy)
    print(f'Student embedding shape: {_s_emb.shape}')

## 6. RKD-only Loss (không có L_KD_cosine)

```
L_total = α·L_MagFace + γ·L_RKD_D + δ·L_RKD_A
```

| Loss | Ý nghĩa |
|---|---|
| `L_MagFace` | Phân biệt class boundary, train student có margin |
| `L_RKD_D` | Pairwise: tỉ lệ khoảng cách giữa các cặp phải tương đồng |
| `L_RKD_A` | Triplet: góc giữa bộ ba phải tương đồng — bảo toàn topology |

**Lý do bỏ L_KD_cosine:** point-wise cosine alignment xung đột với MagFace vì
student MagLinear.weight hội tụ về hướng khác teacher → gradient triệt tiêu nhau.
RKD-D/A bảo toàn *quan hệ tương đối* mà không ràng buộc hướng tuyệt đối.

In [ ]:
class RKDLoss(nn.Module):
    """
    L_total = task_w*L_MagFace + kd_w*L_KD_cosine + rkd_d_w*L_RKD_D + rkd_a_w*L_RKD_A

    Exp27: kd_w=0 → L_KD_cosine disabled.
    RKD-D và RKD-A dùng reduction='mean'.
    """

    def __init__(
        self,
        metadata_path: str,
        task_weight:   float = 2.0,
        kd_weight:     float = 0.0,
        rkd_d_weight:  float = 80.0,
        rkd_a_weight:  float = 160.0,
    ):
        super().__init__()
        self.magface  = WeightClassMagLoss(metadata_path)
        self.task_w   = task_weight
        self.kd_w     = kd_weight
        self.rkd_d_w  = rkd_d_weight
        self.rkd_a_w  = rkd_a_weight

    @staticmethod
    def _pdist(e: torch.Tensor) -> torch.Tensor:
        diff = e.unsqueeze(0) - e.unsqueeze(1)
        return diff.pow(2).sum(-1).clamp(min=1e-12).sqrt()

    def _rkd_distance(self, s_emb: torch.Tensor, t_emb: torch.Tensor) -> torch.Tensor:
        B = s_emb.size(0)
        mask = ~torch.eye(B, dtype=torch.bool, device=s_emb.device)

        with torch.no_grad():
            td   = self._pdist(t_emb)
            mu_t = td[mask].mean()
            td_n = td / (mu_t + 1e-8)

        sd   = self._pdist(s_emb)
        mu_s = sd[mask].mean()
        sd_n = sd / (mu_s + 1e-8)

        return F.huber_loss(sd_n[mask], td_n[mask], delta=1.0, reduction='mean')

    def _rkd_angle(self, s_emb: torch.Tensor, t_emb: torch.Tensor) -> torch.Tensor:
        def _angle_matrix(e: torch.Tensor) -> torch.Tensor:
            diff = e.unsqueeze(0) - e.unsqueeze(1)
            norm = diff.norm(p=2, dim=2, keepdim=True).clamp(min=1e-8)
            diff = diff / norm
            return torch.bmm(diff, diff.transpose(1, 2))

        with torch.no_grad():
            ta = _angle_matrix(t_emb)
        sa = _angle_matrix(s_emb)

        return F.huber_loss(sa, ta, delta=1.0, reduction='mean')

    def forward(
        self,
        student_logits,
        student_norm,
        student_emb,
        teacher_emb,
        id_labels,
    ):
        l_task = self.magface(student_logits, id_labels, student_norm)

        # L_KD_cosine: disabled khi kd_w=0, nhưng vẫn tính để log theo dõi
        with torch.no_grad():
            s_n  = F.normalize(student_emb, p=2, dim=1)
            t_n  = F.normalize(teacher_emb, p=2, dim=1)
            l_kd = (1.0 - F.cosine_similarity(s_n, t_n, dim=1)).mean()

        l_rkd_d = self._rkd_distance(student_emb, teacher_emb)
        l_rkd_a = self._rkd_angle(student_emb, teacher_emb)

        total = (
            self.task_w  * l_task
          + self.rkd_d_w * l_rkd_d
          + self.rkd_a_w * l_rkd_a
        )  # kd_w=0 → L_KD_cosine không tham gia backward
        return total, l_task, l_kd, l_rkd_d, l_rkd_a


criterion = RKDLoss(
    metadata_path=train_csv,
    task_weight=CONFIGURATION['task_weight'],
    kd_weight=CONFIGURATION['kd_weight'],
    rkd_d_weight=CONFIGURATION['rkd_d_weight'],
    rkd_a_weight=CONFIGURATION['rkd_a_weight'],
)
print('RKDLoss (kd_weight=0) khởi tạo thành công.')
print(f"  task={CONFIGURATION['task_weight']}  "
      f"kd={CONFIGURATION['kd_weight']} (disabled)  "
      f"rkd_d={CONFIGURATION['rkd_d_weight']}  "
      f"rkd_a={CONFIGURATION['rkd_a_weight']}")

## 7. Training

In [ ]:
def train_epoch(train_dl, teacher, student, criterion, optimizer, device):
    student.train()

    total_loss = total_task = total_kd = total_rkd_d = total_rkd_a = 0.0

    for X, y in train_dl:
        X, y      = X.to(device), y.to(device)
        id_labels = y[:, 0]

        with torch.no_grad():
            teacher_emb = teacher.get_embedding(X)[-1]

        feat        = student.backbone(X)
        student_emb = student.embedding(feat)
        logits, norm = student.maglinear(student_emb)

        loss, l_task, l_kd, l_rkd_d, l_rkd_a = criterion(
            logits, norm, student_emb, teacher_emb, id_labels
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss  += loss.item()
        total_task  += l_task.item()
        total_kd    += l_kd.item()      # monitored only, not in backward
        total_rkd_d += l_rkd_d.item()
        total_rkd_a += l_rkd_a.item()

    n = len(train_dl)
    return (
        total_loss  / n,
        total_task  / n,
        total_kd    / n,
        total_rkd_d / n,
        total_rkd_a / n,
    )


def display_metrics(epoch, train_metrics, test_metrics):
    rows = []
    for k in train_metrics:
        tv = train_metrics[k]
        ev = test_metrics.get(k, '-')
        fmt = lambda v: f'{v:.4f}' if isinstance(v, float) else str(v)
        rows.append([k, fmt(tv), fmt(ev)])
    print(f'\nEp {epoch}:')
    print(tabulate(rows, headers=['Metric', 'Train', 'Test'], tablefmt='fancy_grid'))

In [ ]:
optimizer = Adam(student.parameters(), lr=CONFIGURATION['base_lr'])
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=20, T_mult=2, eta_min=1e-6)

manager = ExperimentManager(CONFIGURATION)
manager.log_text(
    f"Teacher: {CONFIGURATION['teacher_backbone']} | "
    f"Student: {CONFIGURATION['backbone']} | "
    f"Modality: {CONFIGURATION['type']} | "
    f"task={CONFIGURATION['task_weight']} kd={CONFIGURATION['kd_weight']} (disabled) "
    f"rkd_d={CONFIGURATION['rkd_d_weight']} rkd_a={CONFIGURATION['rkd_a_weight']}"
)

ckpt_saver = ModelCheckpoint(
    output_dir=manager.ckpt_dir,
    mode='max',
    best_metric_name='auc_id_cosine',
)
early_stopping = MultiMetricEarlyStopping(
    monitor_keys=['auc_id_cosine'],
    patience=20,
    mode='max',
    verbose=1,
    save_dir=manager.ckpt_dir,
    start_from_epoch=5,
)

writer = SummaryWriter(log_dir=manager.log_dir)
print(f'Experiment dir: {manager.exp_dir}')
print(f'Checkpoint dir: {manager.ckpt_dir}')
print('(Tất cả được ghi thẳng vào Google Drive)')

In [ ]:
START_EPOCH = 0

manager.log_text('BAT DAU RKD-ONLY KNOWLEDGE DISTILLATION (no KD cosine)')

for epoch in range(START_EPOCH, CONFIGURATION['epochs']):
    manager.log_text(f'\n--- Epoch {epoch+1}/{CONFIGURATION["epochs"]} ---')

    train_loss, train_task, train_kd, train_rkd_d, train_rkd_a = train_epoch(
        train_dl, teacher, student, criterion, optimizer, device
    )

    train_auc = compute_id_auc(train_dl, student, device)
    test_auc  = compute_id_auc(test_dl,  student, device)

    train_metrics = {
        'loss':             train_loss,
        'loss_task':        train_task,
        'loss_kd (monitor)': train_kd,  # không tham gia backward, chỉ theo dõi
        'loss_rkd_d':       train_rkd_d,
        'loss_rkd_a':       train_rkd_a,
        'auc_id_cosine':    train_auc['id_cosine'],
        'auc_id_euclidean': train_auc['id_euclidean'],
    }
    test_metrics = {
        'auc_id_cosine':    test_auc['id_cosine'],
        'auc_id_euclidean': test_auc['id_euclidean'],
    }

    writer.add_scalar('Loss/total',          train_loss,   epoch + 1)
    writer.add_scalar('Loss/task',           train_task,   epoch + 1)
    writer.add_scalar('Loss/kd_monitor',     train_kd,     epoch + 1)
    writer.add_scalar('Loss/rkd_d',          train_rkd_d,  epoch + 1)
    writer.add_scalar('Loss/rkd_a',          train_rkd_a,  epoch + 1)
    writer.add_scalars('AUC/cosine',
        {'train': train_auc['id_cosine'],    'test': test_auc['id_cosine']},    epoch + 1)
    writer.add_scalars('AUC/euclidean',
        {'train': train_auc['id_euclidean'], 'test': test_auc['id_euclidean']}, epoch + 1)

    display_metrics(epoch + 1, train_metrics, test_metrics)
    manager.log_metrics(epoch + 1, {**train_metrics, **test_metrics})

    ckpt_saver(student, optimizer, epoch + 1, test_metrics, scheduler)
    early_stopping(test_metrics, student, epoch + 1)
    scheduler.step(epoch)

    if early_stopping.early_stop:
        manager.log_text('Early stopping triggered.')
        break

writer.close()
manager.log_text('RKD-ONLY KD HOAN TAT.')

## 8. Resume Training từ Checkpoint

> Chạy cell này khi Colab disconnect và muốn tiếp tục train.  
> **Cách dùng:** Chạy cell Setup → Imports → Data → Teacher → Student → Loss → Setup Train,  
> sau đó chạy cell này, rồi chạy lại cell fit.

In [ ]:
CKPT_PATH = os.path.join(manager.ckpt_dir, 'last_model.pth')

if not os.path.exists(CKPT_PATH):
    raise FileNotFoundError(f'Không tìm thấy checkpoint: {CKPT_PATH}')

checkpoint = torch.load(CKPT_PATH, map_location=device)
student.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
if 'scheduler_state_dict' in checkpoint:
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])

START_EPOCH = checkpoint['epoch']
print(f'Resume từ epoch {START_EPOCH} — chạy lại cell "cell-fit" để tiếp tục.')

## 9. Đánh giá Final

In [ ]:
best_ckpt_path = os.path.join(manager.ckpt_dir, 'best_model.pth')
best_ckpt = torch.load(best_ckpt_path, map_location=device, weights_only=False)
student.load_state_dict(best_ckpt['model_state_dict'])
student.eval()
print(f"Best model từ epoch {best_ckpt.get('epoch', '?')}")

student_gp_auc   = compute_id_auc_gallery_probe(gallery_dl, probe_dl, student, device)
student_gp_rank1 = compute_rank1_gallery_probe(gallery_dl, probe_dl, student, device)

rows = [
    ['Cosine AUC    (gallery→probe)', f"{student_gp_auc['id_cosine']:.4f}"],
    ['Euclidean AUC (gallery→probe)', f"{student_gp_auc['id_euclidean']:.4f}"],
    ['Rank-1 Acc    (gallery→probe)', f"{student_gp_rank1:.4f}"],
]
print(f"\nStudent ({CONFIGURATION['backbone']}) — RKD-only from {CONFIGURATION['teacher_backbone']}")
print(tabulate(rows, headers=['Metric', 'Value'], tablefmt='fancy_grid'))

## 9.1. So sánh Teacher vs Student vs Exp26

In [ ]:
# Kết quả Exp26 (task=2 kd=1 rkd_d=1 rkd_a=2) để so sánh
exp26_rank1 = 0.78   # cập nhật lại nếu có số chính xác
exp26_auc_cosine = '-'

compare_rows = [
    ['Model',                         CONFIGURATION['teacher_backbone'],          CONFIGURATION['backbone'] + ' (Exp26)',   CONFIGURATION['backbone'] + ' (Exp27)'],
    ['Config',                        'MTL+FSM+GRL',                              'task=2 kd=1 rkd_d=1 rkd_a=2',           'task=2 kd=0 rkd_d=80 rkd_a=160'],
    ['Params',                        f'{teacher_params:,}',                      f'{total_p:,}',                           f'{total_p:,}'],
    ['Cosine AUC (gallery→probe)',     f"{teacher_gp_auc['id_cosine']:.4f}",       exp26_auc_cosine,                         f"{student_gp_auc['id_cosine']:.4f}"],
    ['Euclidean AUC (gallery→probe)',  f"{teacher_gp_auc['id_euclidean']:.4f}",    '-',                                      f"{student_gp_auc['id_euclidean']:.4f}"],
    ['Rank-1 Acc (gallery→probe)',     f"{teacher_gp_rank1:.4f}",                 f"{exp26_rank1:.4f}",                     f"{student_gp_rank1:.4f}"],
]
print(tabulate(compare_rows, headers=['Metric', 'Teacher', 'Student Exp26', 'Student Exp27 (RKD-only)'], tablefmt='fancy_grid'))

## 10. Export ONNX

Export phần inference (backbone + embedding + L2 normalize, bỏ MagLinear) để chuẩn bị quantize INT8 deploy edge device.

In [ ]:
class InferenceWrapper(nn.Module):
    """Backbone + embedding + L2 normalize — không có MagLinear."""
    def __init__(self, model):
        super().__init__()
        self.backbone  = model.backbone
        self.embedding = model.embedding

    def forward(self, x):
        emb = self.embedding(self.backbone(x))
        return F.normalize(emb, p=2, dim=1)


inference_model = InferenceWrapper(student).eval().cpu()
dummy_input = torch.randn(1, 3, 112, 112)

onnx_path = os.path.join(manager.ckpt_dir, 'rkd_only_mobilenetv3_fr.onnx')

torch.onnx.export(
    inference_model,
    dummy_input,
    onnx_path,
    input_names=['input'],
    output_names=['embedding'],
    dynamic_axes={'input': {0: 'batch'}, 'embedding': {0: 'batch'}},
    opset_version=17,
)
print(f'Exported ONNX: {onnx_path}')